# Exercise pose classification with PyTorch

This notebook follows a typical [Kaggle](https://www.kaggle.com/code/teowaihong/exercise-pose-classification-with-pytorch)-style workflow:

1. Load **MediaPipe-style landmarks** from the [Physical Exercise Recognition](https://www.kaggle.com/datasets/muhannadtuameh/exercise-recognition) dataset (`landmarks.csv` + `labels.csv`).
2. Build **sliding windows** of 2D pose features (COCO-17 xy after mapping from 33 landmarks).
3. Train a small **LSTM** classifier for **exercise / phase** labels.

**On Kaggle:** Add the dataset `muhannadtuameh/exercise-recognition` as an input — it appears under `/kaggle/input/exercise-recognition/`.

**Locally:** Set `DATA_ROOT` below to your extracted folder (or run `kaggle_exercise_recognition_pipeline.py` in this repo first).

## 1. Setup

In [2]:
from __future__ import annotations

import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset

# Reproducibility
def set_seed(s: int = 42) -> None:
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


## 2. Resolve data path (Kaggle vs local)

- **Kaggle:** any folder under `/kaggle/input/` that contains `landmarks.csv`.
- **Local:** (1) env `EXERCISE_RECOGNITION_ROOT`, (2) `results/kaggle_exercise_recognition/` by walking **up** from cwd (works when the kernel cwd is `notebooks/`), (3) `~/.cache/kagglehub/.../exercise-recognition/versions/*/`.
- If it still fails: `export EXERCISE_RECOGNITION_ROOT=/path/to/dataset` or set the notebook kernel’s working directory to the **project root**.

In [3]:
import os


def resolve_data_root() -> Path:
    # Explicit override (best for local Jupyter)
    env = os.environ.get("EXERCISE_RECOGNITION_ROOT", "").strip()
    if env:
        p = Path(env).expanduser().resolve()
        if (p / "landmarks.csv").is_file():
            return p

    # Kaggle: any input dir that contains landmarks.csv
    kin = Path("/kaggle/input")
    if kin.is_dir():
        for sub in sorted(kin.iterdir()):
            if sub.is_dir() and (sub / "landmarks.csv").is_file():
                return sub

    # Local: walk cwd and parents for repo-style results/ (works if cwd is notebooks/ or repo root)
    start = Path.cwd().resolve()
    for base in [start, *list(start.parents)[:8]]:
        c = base / "results" / "kaggle_exercise_recognition"
        if (c / "landmarks.csv").is_file():
            return c.resolve()

    # kagglehub default cache (any version folder)
    hub = Path.home() / ".cache/kagglehub/datasets/muhannadtuameh/exercise-recognition"
    if hub.is_dir():
        for v in sorted(hub.glob("versions/*")):
            if v.is_dir() and (v / "landmarks.csv").is_file():
                return v.resolve()

    raise FileNotFoundError(
        "Could not find landmarks.csv.\n"
        "  • export EXERCISE_RECOGNITION_ROOT=/path/to/folder  (must contain landmarks.csv + labels.csv)\n"
        "  • Or run from the repo: python kaggle_exercise_recognition_pipeline.py --download\n"
        "  • Or open Jupyter from repo root / set kernel cwd to project root so results/kaggle_exercise_recognition is found"
    )


DATA_ROOT = resolve_data_root()
print("DATA_ROOT =", DATA_ROOT)

FileNotFoundError: Could not find landmarks.csv. On Kaggle, add dataset muhannadtuameh/exercise-recognition. Locally, download and point to the folder containing landmarks.csv + labels.csv.

## 3. Load CSVs → COCO-17 xy (same mapping as capstone repo)

BlazePose 33 → COCO-17 index list (copy of `pose_estimation_core.MEDIAPIPE_TO_COCO`).

In [ ]:
MEDIAPIPE_TO_COCO = (
    0, 2, 5, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28
)


def load_landmarks_csv(path: Path) -> tuple[np.ndarray, np.ndarray]:
    df = pd.read_csv(path)
    cols = [c for c in df.columns if c != "pose_id"]
    assert len(cols) == 99, f"expected 33×3 xyz columns, got {len(cols)}"
    arr = df[cols].to_numpy(dtype=np.float64)
    kp33 = arr.reshape(len(df), 33, 3)
    return kp33, df["pose_id"].to_numpy()


def mediapipe33_to_coco17_xy(kp33: np.ndarray) -> np.ndarray:
    t = kp33.shape[0]
    out = np.zeros((t, 17, 2), dtype=np.float64)
    for coco_i, mp_i in enumerate(MEDIAPIPE_TO_COCO):
        out[:, coco_i, :] = kp33[:, mp_i, :2]
    return out


kp33, pose_ids = load_landmarks_csv(DATA_ROOT / "landmarks.csv")
labels_df = pd.read_csv(DATA_ROOT / "labels.csv")
assert len(labels_df) == len(kp33), "landmarks / labels row mismatch"

X_flat = mediapipe33_to_coco17_xy(kp33).reshape(len(kp33), -1)  # (T, 34)
y_text = labels_df["pose"].astype(str).values
print("frames:", X_flat.shape[0], "feat_dim:", X_flat.shape[1], "unique labels:", len(np.unique(y_text)))

## 4. Sliding windows + label encoder

Each window uses the **last frame’s** label (simple; you can switch to majority vote).

In [ ]:
WINDOW = 16
STRIDE = 8


def build_windows(feats: np.ndarray, labels: np.ndarray, window: int, stride: int) -> tuple[np.ndarray, np.ndarray]:
    xs, ys = [], []
    t = feats.shape[0]
    for start in range(0, t - window + 1, stride):
        end = start + window
        xs.append(feats[start:end])
        ys.append(labels[end - 1])
    return np.stack(xs, axis=0), np.array(ys)


X_win, y_win = build_windows(X_flat, y_text, WINDOW, STRIDE)
le = LabelEncoder()
y_idx = le.fit_transform(y_win)
num_classes = len(le.classes_)
print("windows:", X_win.shape, "classes:", num_classes)
print("class names (sample):", list(le.classes_)[:10])

## 5. PyTorch `Dataset` + train/val split

In [ ]:
from typing import Optional


class PoseWindowDataset(Dataset):
    def __init__(
        self,
        x: np.ndarray,
        y: np.ndarray,
        mean: Optional[np.ndarray] = None,
        std: Optional[np.ndarray] = None,
    ):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.mean = mean
        self.std = std

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, i: int) -> tuple[torch.Tensor, torch.Tensor]:
        w = self.x[i].clone()
        if self.mean is not None and self.std is not None:
            w = (w - self.mean) / (self.std + 1e-8)
        return w, self.y[i]


X_train, X_val, y_train, y_val = train_test_split(X_win, y_idx, test_size=0.2, random_state=42, stratify=y_idx)
mean = X_train.mean(axis=(0, 1))
std = X_train.std(axis=(0, 1))

train_ds = PoseWindowDataset(X_train, y_train, mean, std)
val_ds = PoseWindowDataset(X_val, y_val, mean, std)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
print("train batches:", len(train_loader), "val batches:", len(val_loader))

## 6. LSTM classifier

In [ ]:
class PoseLSTM(nn.Module):
    def __init__(self, input_dim: int, hidden: int, num_classes: int, num_layers: int = 1, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        last = self.drop(last)
        return self.fc(last)


feat_dim = X_win.shape[2]
model = PoseLSTM(feat_dim, hidden=64, num_classes=num_classes, num_layers=1, dropout=0.2).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
print(model)

## 7. Train & validate

In [ ]:
EPOCHS = 30


def accuracy(logits: torch.Tensor, y: torch.Tensor) -> float:
    pred = logits.argmax(dim=1)
    return (pred == y).float().mean().item()


for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    tr_acc = 0.0
    n = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        opt.step()
        tr_loss += loss.item() * len(yb)
        tr_acc += accuracy(logits, yb) * len(yb)
        n += len(yb)
    tr_loss /= n
    tr_acc /= n

    model.eval()
    va_loss = 0.0
    va_acc = 0.0
    n = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            va_loss += loss.item() * len(yb)
            va_acc += accuracy(logits, yb) * len(yb)
            n += len(yb)
    va_loss /= n
    va_acc /= n

    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(f"epoch {epoch:02d}  train loss {tr_loss:.4f} acc {tr_acc:.4f}  |  val loss {va_loss:.4f} acc {va_acc:.4f}")

print("Done.")

## 8. (Optional) Save weights — Kaggle output

On Kaggle, write to `/kaggle/working/`.

In [ ]:
out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
ckpt = out_dir / "pose_lstm_exercise_recognition.pt"
torch.save({"model": model.state_dict(), "classes": le.classes_.tolist(), "mean": mean, "std": std, "window": WINDOW}, ckpt)
print("saved:", ckpt.resolve())

---

### Using this repo’s pipeline instead of raw CSV

After running `kaggle_exercise_recognition_pipeline.py`, you can load `kaggle_exercise_recognition_biomechanics.npz` (8 joint angles × T) and repeat the same windowing + LSTM on **angle features** instead of flattened xy — swap `X_flat` for angles and set `feat_dim = 8`.